In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-07 02:34:06.043494: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-07 02:34:06.694812: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-07 02:34:07,766 [DEBUG] [Rain] Rain is initialized
2023-07-07 02:34:07,769 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 02:34:07,770 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 02:34:07,771 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 02:34:07,772 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 02:34:07,774 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 02:34:07,775 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 02:34:07,776 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-07 02:34:07,787 [INFO] [Provisioner] provisioner is serving
2023-07-07 02:34:07,788 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 02:34:07,790 [INFO] [Coordinator] coordinator is serving
2023-07-07 02:34:07,791 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 02:34:07,795 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 02:34:07,796 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 02:34:07,797 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 02:34:07,798 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 02:34:07,800 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-07 02:34:07,801 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50152/
2023-07-07 02:34:07,803 [INFO] [Worker_50152] Worker is running on port: 50152
2023-0

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 2s 6ms/step - loss: 0.7006 - accuracy: 0.7818
Epoch 2/5
157/157 [==============================] - 2s 7ms/step - loss: 0.6982 - accuracy: 0.7828
Epoch 2/5
157/157 [==============================] - 1s 7ms/step - loss: 0.3081 - accuracy: 0.9099
Epoch 3/5
157/157 [==============================] - 1s 7ms/step - loss: 0.3044 - accuracy: 0.9067
Epoch 3/5
157/157 [==============================] - 1s 7ms/step - loss: 0.3095 - accuracy: 0.9069
Epoch 3/5
157/157 [==============================] - 1s 7ms/step - loss: 0.2345 - accuracy: 0.9299
Epoch 4/5
157/157 [==============================] - 1s 7ms/step - loss: 0.2308 - accuracy: 0.9299
Epoch 4/5
157/157 [==============================] - 1s 7ms/step - loss: 0.2325 - accuracy: 0.9298
Epoch 4/5
157/157 [==============================] - 1s 7ms/step - loss: 0.1916 - accuracy: 0.9414
Epoch 5/5
157/157 [==============================] - 1s 7ms/step - loss: 0.1889 - accurac

2023-07-07 02:34:17,049 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 02:34:17,052 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider
152/157 [============================>.] - ETA: 0s - loss: 0.1637 - accuracy: 0.9490

2023-07-07 02:34:17,133 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-07 02:34:17,152 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1


...

2023-07-07 02:34:30,874 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 1.
2023-07-07 02:34:30,877 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 02:34:30,888 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3


157/157 [==============================] - 1s 6ms/step - loss: 0.0809 - accuracy: 0.9736


DEBUG:DeepLearning:Asynchronous update is done by worker 3
2023-07-07 02:34:30,903 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-07 02:34:30,905 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to divider


2023-07-07 02:34:30,913 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 3.
2023-07-07 02:34:30,958 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-07 02:34:30,965 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
DEBUG:DeepLearning:Asynchronous update is done by worker 2
2023-07-07 02:34:30,985 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 2.
2023-07-07 02:34:30,987 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-07 02:34:30,988 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0834 - accuracy: 0.9772

Test accuracy: 97.7%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 02:34:31,346 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-07 02:34:31,347 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-07 02:34:31,350 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-07 02:34:31,351 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-07 02:34:31,353 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 02:34:31,355 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-07 02:34:31,356 [DEBUG] [LocalProvisioner] Creating 3 workers
DEBUG:LocalProvisioner:Creating 3 

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 3s 7ms/step - loss: 0.1076 - accuracy: 0.9685
Epoch 2/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0881 - accuracy: 0.9727
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0990 - accuracy: 0.9683
Epoch 3/5
157/157 [==============================] - 1s 8ms/step - loss: 0.0807 - accuracy: 0.9761
Epoch 4/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0834 - accuracy: 0.9725
Epoch 4/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0867 - accuracy: 0.9726
Epoch 4/5
157/157 [==============================] - 1s 7ms/step - loss: 0.0744 - accuracy: 0.9764
Epoch 5/5
157/157 [==============================] - 1s 7ms/step - loss: 0.0767 - accuracy: 0.9754
Epoch 5/5
157/157 [==============================] - 1s 7ms/step - loss: 0.0798 - accuracy: 0.9746
Epoch 5/5
157/157 [==============================] - 1s 7ms/step - loss: 0.0701 - accurac

2023-07-07 02:34:41,601 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 02:34:41,605 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3


sending data to divider
152/157 [============================>.] - ETA: 0s - loss: 0.0735 - accuracy: 0.9757

2023-07-07 02:34:41,634 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 02:34:41,636 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


...

2023-07-07 02:34:55,240 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 02:34:55,278 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully


157/157 [==============================] - 1s 5ms/step - loss: 0.0520 - accuracy: 0.9822


2023-07-07 02:34:55,379 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-07 02:34:55,380 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2


sending data to divider


2023-07-07 02:34:55,432 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-07 02:34:55,454 [DEBUG] [DeepLearning] Iteration 3/3 complete.
DEBUG:DeepLearning:Iteration 3/3 complete.
2023-07-07 02:34:55,456 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-07 02:34:55,457 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving


In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0685 - accuracy: 0.9814

Test accuracy: 98.1%
